In [ ]:
!pip install librosa --quiet

In [ ]:
import os, numpy as np, librosa, tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("carlthome/gtzan-genre-collection")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'gtzan-genre-collection' dataset.
Path to dataset files: /kaggle/input/gtzan-genre-collection


In [ ]:
import os

data_dir = '/kaggle/input/gtzan-genre-collection'
os.listdir(data_dir)


['genres']

In [ ]:
data_dir = '/kaggle/input/gtzan-genre-collection/genres'


In [ ]:
def extract_features(file_path, max_pad_len=130):
    try:
        audio, sr = librosa.load(file_path, res_type='kaiser_fast', duration=30)
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        pad_width = max_pad_len - mfccs.shape[1]
        if pad_width > 0:
            mfccs = np.pad(mfccs, pad_width=((0,0),(0,pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]
        return mfccs
    except Exception as e:
        print("Error:", file_path, e)
        return None

X, y, labels = [], [], os.listdir(data_dir)
print("Extracting MFCCs...")
for i, label in enumerate(labels):
    folder = os.path.join(data_dir, label)
    for f in tqdm(os.listdir(folder), desc=label):
        fp = os.path.join(folder, f)
        features = extract_features(fp)
        if features is not None:
            X.append(features)
            y.append(i)

X, y = np.array(X), np.array(y)
print("✅ Features shape:", X.shape)

Extracting MFCCs...


pop: 100%|██████████| 100/100 [00:04<00:00, 21.45it/s]


✅ Features shape: (1000, 40, 130)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Reshape the input data to be 3D for the LSTM layer - This reshaping is redundant
# X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2])
# X_test  = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2])
y_train, y_test = to_categorical(y_train), to_categorical(y_test)

In [ ]:
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.5),
    LSTM(64, return_sequences=True),
    Dropout(0.5),
    LSTM(32),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(len(labels), activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_11 (LSTM)                  │ (None, 40, 128)        │       132,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 40, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ (None, 40, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 40, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_13 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 197,194 (770.29 KB)

 Trainable params: 197,194 (770.29 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.0001)

history = model.fit(X_train, y_train, validation_data=(X_test, y_test),
                    epochs=50, batch_size=32, callbacks=[early_stopping, reduce_lr])

Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - accuracy: 0.1039 - loss: 2.3124 - val_accuracy: 0.0950 - val_loss: 2.3059 - learning_rate: 0.0010
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.1474 - loss: 2.2777 - val_accuracy: 0.1250 - val_loss: 2.2786 - learning_rate: 0.0010
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.1471 - loss: 2.2443 - val_accuracy: 0.1350 - val_loss: 2.2697 - learning_rate: 0.0010
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.1833 - loss: 2.2067 - val_accuracy: 0.1550 - val_loss: 2.2436 - learning_rate: 0.0010
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.2171 - loss: 2.1498 - val_accuracy: 0.1950 - val_loss: 2.1874 - learning_rate: 0.0010
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2700 - loss: 2.0589 - val_accuracy: 0.1750 - val_loss: 2.1719 - learning_rate: 0.0010
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2970 - loss: 1.9468 - val_acc

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print(f"\n✅ Test Accuracy: {acc*100:.2f}%")

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2543 - loss: 2.0734

✅ Test Accuracy: 25.00%
